# ST-GCN joint_bone - fixing the diagnosed premature LR freeze

The `joint_bone` run plateaued from epoch 20 onward - `train_acc` and `val_top1` both flat and oscillating, not trending up. The cause: `StepLR(step_size=10, gamma=0.1)` crushes the learning rate to near-zero by epoch 30-40, well before the 60-epoch budget ends. The model genuinely can't learn much more past that point with this schedule, regardless of epoch count.

This tests stretching the decay schedule out - `step_size=15` and `step_size=20` - so the model spends more real epochs at each usable learning rate before it dies. Same architecture (`joint_bone`, `in_channels=5`), same `weight_decay=1e-3`, same advanced augmentation, same seeded-worker fix from the last notebook. `step_size=10` is re-run here too as a fixed reference point, under identical seeding to the other two - the previous 72.63% result used the same step_size=10 but without full worker-seeding control, so this gives a clean three-way comparison rather than comparing against a differently-seeded run.

### determinism - before torch touches CUDA

In [1]:
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

import sys
import json
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from sklearn.metrics import (
    accuracy_score, f1_score, balanced_accuracy_score, top_k_accuracy_score,
    confusion_matrix, precision_recall_fscore_support,
)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

sys.path.insert(0, "/workspace/badminton-honours-project")
import dataset as ds

import mmaction.models  # noqa: F401
from mmaction.utils import register_all_modules
register_all_modules()
from mmaction.registry import MODELS

from clearml import Task

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("device:", device)


device: cuda:0


### DataLoader worker-seeding fix - same as the last notebook

In [2]:
def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)


### paths

In [3]:
OUT_ROOT = "/workspace/outputs/stgcn_jointbone_schedule"
CKPT_ROOT = "/workspace/checkpoints/stgcn_jointbone_schedule"
os.makedirs(OUT_ROOT, exist_ok=True)
os.makedirs(CKPT_ROOT, exist_ok=True)


### data - advanced augmentation

In [4]:
manifest = ds.load_filtered_manifest(
    "/workspace/data/split_manifest.csv",
    "/workspace/data/extraction_log.csv",
)

class_names = sorted(manifest["class_name"].unique())
class_to_id = {name: i for i, name in enumerate(class_names)}
num_classes = len(class_names)

train_manifest = manifest[manifest["split"] == "train"].reset_index(drop=True)
val_manifest = manifest[manifest["split"] == "val"].reset_index(drop=True)
test_manifest = manifest[manifest["split"] == "test"].reset_index(drop=True)

KEYPOINTS_ROOT = "/workspace/data/keypoints_smoothed"

train_ds = ds.BadmintonPoseDataset(train_manifest, KEYPOINTS_ROOT, class_to_id, augmentation_policy="advanced")
val_ds = ds.BadmintonPoseDataset(val_manifest, KEYPOINTS_ROOT, class_to_id)
test_ds = ds.BadmintonPoseDataset(test_manifest, KEYPOINTS_ROOT, class_to_id)

balanced_sampler = ds.build_class_balanced_sampler(train_manifest, target_per_class=500, seed=42)

val_loader = DataLoader(val_ds, batch_size=64, shuffle=False, num_workers=4, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=64, shuffle=False, num_workers=4, pin_memory=True)

print(f"{num_classes} classes, train {len(train_manifest)}, val {len(val_ds)}, test {len(test_ds)}")


dropped 4 manually-excluded clips from the manifest (7814 remain)
18 classes, train 6251, val 781, test 782


### batch -> ST-GCN input - joint_bone only, in_channels=5

In [5]:
def batch_to_stgcn_input(batch):
    x = torch.cat([batch["keypoints"].float(), batch["bone"].float()], dim=-1)
    return x.unsqueeze(1).unsqueeze(1)   # (N, num_clips=1, M=1, T, V, C=5)


### SMOKE TEST - run alone first

In [6]:
smoke_loader = DataLoader(train_ds, batch_size=4, shuffle=True)
smoke_batch = next(iter(smoke_loader))
smoke_input = batch_to_stgcn_input(smoke_batch)
print("input shape:", smoke_input.shape)
assert smoke_input.shape[-1] == 5, f"expected 5 channels, got {smoke_input.shape[-1]}"
print("smoke test passed")


input shape: torch.Size([4, 1, 1, 30, 17, 5])
smoke test passed


### metrics

In [7]:
@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    all_labels, all_preds, all_probs = [], [], []

    for batch in loader:
        x = batch_to_stgcn_input(batch).to(device)
        y = batch["label"].to(device)
        logits = model(x, mode="tensor", stage="head")
        probs = torch.softmax(logits, dim=1)

        all_labels.append(y.cpu().numpy())
        all_preds.append(probs.argmax(dim=1).cpu().numpy())
        all_probs.append(probs.cpu().numpy())

    labels = np.concatenate(all_labels)
    preds = np.concatenate(all_preds)
    probs = np.concatenate(all_probs)

    return {
        "top1": accuracy_score(labels, preds),
        "top5": top_k_accuracy_score(labels, probs, k=5, labels=list(range(num_classes))),
        "mca": balanced_accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro"),
        "labels": labels, "preds": preds,
    }


### the reusable training function

Fixed: joint_bone architecture, weight_decay=1e-3, advanced augmentation, seeded workers. Only `step_size` varies between calls.

In [8]:
def train_one_schedule(step_size, seed=42, num_epochs=60, patience=20, batch_size=16,
                        lr=0.01, weight_decay=1e-3):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)

    model_cfg = dict(
        type="RecognizerGCN",
        backbone=dict(
            type="STGCN",
            graph_cfg=dict(layout="coco", mode="stgcn_spatial"),
            in_channels=5, base_channels=64, ch_ratio=2,
            num_stages=9, inflate_stages=[4, 7], down_stages=[4, 7],
        ),
        cls_head=dict(type="GCNHead", num_classes=num_classes, in_channels=256,
                      loss_cls=dict(type="CrossEntropyLoss")),
    )
    model = MODELS.build(model_cfg).to(device)

    loader_generator = torch.Generator()
    loader_generator.manual_seed(seed)

    train_loader = DataLoader(
        train_ds, batch_size=batch_size, sampler=balanced_sampler,
        num_workers=4, pin_memory=True, persistent_workers=True,
        worker_init_fn=seed_worker, generator=loader_generator,
    )

    config_name = f"step{step_size}"
    task = Task.init(project_name="BadmintonPoseAnalysis", task_name=f"stgcn_jointbone_{config_name}", reuse_last_task_id=False)
    logger = task.get_logger()
    task.connect({
        "stream": "joint_bone", "step_size": step_size, "seed": seed,
        "lr": lr, "weight_decay": weight_decay, "num_epochs": num_epochs,
        "batch_size": batch_size, "augmentation_policy": "advanced",
    })

    optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=weight_decay)
    criterion = nn.CrossEntropyLoss()
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=step_size, gamma=0.1)

    best_val_top1 = 0.0
    best_epoch = -1
    epochs_without_improvement = 0
    best_state = None

    for epoch in range(num_epochs):
        model.train()
        epoch_loss = 0.0
        epoch_labels, epoch_preds = [], []

        for batch in train_loader:
            x = batch_to_stgcn_input(batch).to(device)
            y = batch["label"].to(device)

            optimizer.zero_grad()
            logits = model(x, mode="tensor", stage="head")
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item() * x.size(0)
            epoch_labels.append(y.cpu().numpy())
            epoch_preds.append(logits.argmax(dim=1).cpu().numpy())

        scheduler.step()

        train_loss = epoch_loss / len(train_loader.dataset)
        train_acc = accuracy_score(np.concatenate(epoch_labels), np.concatenate(epoch_preds))
        val_metrics = evaluate(model, val_loader)

        logger.report_scalar("loss", "train", train_loss, iteration=epoch)
        logger.report_scalar("accuracy", "train", train_acc, iteration=epoch)
        logger.report_scalar("accuracy", "val_top1", val_metrics["top1"], iteration=epoch)
        logger.report_scalar("accuracy", "val_mca", val_metrics["mca"], iteration=epoch)
        logger.report_scalar("lr", "value", optimizer.param_groups[0]["lr"], iteration=epoch)

        if val_metrics["top1"] > best_val_top1:
            best_val_top1 = val_metrics["top1"]
            best_epoch = epoch
            epochs_without_improvement = 0
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            epochs_without_improvement += 1

        if epoch % 10 == 0 or epoch == num_epochs - 1:
            lr_now = optimizer.param_groups[0]["lr"]
            print(f"[{config_name}] epoch {epoch:3d}  lr={lr_now:.5f}  train_acc={train_acc:.4f}  "
                  f"val_top1={val_metrics['top1']:.4f}  val_mca={val_metrics['mca']:.4f}")

        if epochs_without_improvement >= patience:
            print(f"[{config_name}] early stopping at epoch {epoch}, best val_top1={best_val_top1:.4f} (epoch {best_epoch})")
            break

    model.load_state_dict(best_state)

    train_final = evaluate(model, DataLoader(train_ds, batch_size=64, shuffle=False, num_workers=4, pin_memory=True))
    test_final = evaluate(model, test_loader)

    metrics = {
        "config_name": config_name,
        "step_size": step_size,
        "best_epoch": best_epoch,
        "train_acc": train_final["top1"],
        "test_top1": test_final["top1"],
        "test_top5": test_final["top5"],
        "test_mca": test_final["mca"],
        "test_macro_f1": test_final["macro_f1"],
        "train_test_gap": train_final["top1"] - test_final["top1"],
    }

    for metric_name in ["train_acc", "test_top1", "test_top5", "test_mca", "test_macro_f1", "train_test_gap", "best_epoch"]:
        logger.report_single_value(name=metric_name, value=metrics[metric_name])

    cm = confusion_matrix(test_final["labels"], test_final["preds"], labels=list(range(num_classes)))
    logger.report_confusion_matrix(
        title="Test Confusion Matrix", series=config_name,
        matrix=cm, xaxis="Predicted", yaxis="Actual",
        xlabels=class_names, ylabels=class_names,
    )

    precision, recall, f1, support = precision_recall_fscore_support(
        test_final["labels"], test_final["preds"], labels=list(range(num_classes)), zero_division=0,
    )
    per_class_df = pd.DataFrame({
        "class": class_names, "precision": precision, "recall": recall,
        "f1": f1, "support": support,
    })
    logger.report_table(title="Per-Class Test Metrics", series=config_name, table_plot=per_class_df)

    ckpt_path = os.path.join(CKPT_ROOT, f"{config_name}_best.pt")
    torch.save(model.state_dict(), ckpt_path)
    task.upload_artifact("checkpoint", ckpt_path)
    per_class_df.to_csv(os.path.join(OUT_ROOT, f"per_class_{config_name}.csv"), index=False)
    task.close()

    return metrics


### run all three step_size values

3x a single run. `step_size=10` is a fresh, fully-seeded re-run of the original schedule for a clean baseline, not a reused number from before.

In [9]:
step_sizes_to_test = [10, 15, 20]

all_results = {}

for step_size in step_sizes_to_test:
    print(f"\n=== training step_size={step_size} ===")
    metrics = train_one_schedule(step_size)
    all_results[metrics["config_name"]] = metrics
    print(f"[{metrics['config_name']}] FINAL: best_epoch={metrics['best_epoch']}  "
          f"test_top1={metrics['test_top1']:.4f}  gap={metrics['train_test_gap']:.4f}  "
          f"test_mca={metrics['test_mca']:.4f}  test_macro_f1={metrics['test_macro_f1']:.4f}")



=== training step_size=10 ===
ClearML Task: created new task id=aa53e0bdb11448f9b0ec73eeeeb386c9
2026-09-10 08:16:17,223 - clearml.Repository Detection - WARNING - Failed accessing the jupyter server(s): []
ClearML results page: https://app.clear.ml/projects/89838577f7964905a3b492353933fcb2/tasks/aa53e0bdb11448f9b0ec73eeeeb386c9/output/log
[step10] epoch   0  lr=0.01000  train_acc=0.2898  val_top1=0.5506  val_mca=0.3714
[step10] epoch  10  lr=0.00100  train_acc=0.7645  val_top1=0.6761  val_mca=0.6453
[step10] epoch  20  lr=0.00010  train_acc=0.8910  val_top1=0.7093  val_mca=0.6425
[step10] epoch  30  lr=0.00001  train_acc=0.9142  val_top1=0.7196  val_mca=0.6293
[step10] epoch  40  lr=0.00000  train_acc=0.9167  val_top1=0.7196  val_mca=0.6348
[step10] early stopping at epoch 47, best val_top1=0.7324 (epoch 27)


███████████████████████████████ 100% | 11.66/11.66 MB [00:01<00:00,  8.24MB/s]: 


[step10] FINAL: best_epoch=27  test_top1=0.7110  gap=0.2161  test_mca=0.6160  test_macro_f1=0.6302

=== training step_size=15 ===


Failed accessing the jupyter server(s): []


ClearML Task: created new task id=d21b84712fbc4dada4b7b865cfb3f0da
ClearML results page: https://app.clear.ml/projects/89838577f7964905a3b492353933fcb2/tasks/d21b84712fbc4dada4b7b865cfb3f0da/output/log
[step15] epoch   0  lr=0.01000  train_acc=0.3041  val_top1=0.4981  val_mca=0.3976
[step15] epoch  10  lr=0.01000  train_acc=0.7071  val_top1=0.6620  val_mca=0.6156
[step15] epoch  20  lr=0.00100  train_acc=0.9025  val_top1=0.7362  val_mca=0.6465
[step15] epoch  30  lr=0.00010  train_acc=0.9629  val_top1=0.7414  val_mca=0.6274
[step15] epoch  40  lr=0.00010  train_acc=0.9789  val_top1=0.7388  val_mca=0.6296
[step15] epoch  50  lr=0.00001  train_acc=0.9818  val_top1=0.7503  val_mca=0.6448
[step15] epoch  59  lr=0.00000  train_acc=0.9835  val_top1=0.7465  val_mca=0.6374


...

[step15] FINAL: best_epoch=50  test_top1=0.7097  gap=0.2842  test_mca=0.5902  test_macro_f1=0.6116

=== training step_size=20 ===


Failed accessing the jupyter server(s): []


ClearML Task: created new task id=3427eaca1ba8433c917a4d7d311a8842
ClearML results page: https://app.clear.ml/projects/89838577f7964905a3b492353933fcb2/tasks/3427eaca1ba8433c917a4d7d311a8842/output/log
[step20] epoch   0  lr=0.01000  train_acc=0.2962  val_top1=0.4123  val_mca=0.3372
[step20] epoch  10  lr=0.01000  train_acc=0.7022  val_top1=0.6517  val_mca=0.6084
[step20] epoch  20  lr=0.00100  train_acc=0.8871  val_top1=0.7081  val_mca=0.6378
[step20] epoch  30  lr=0.00100  train_acc=0.9709  val_top1=0.7209  val_mca=0.6333
[step20] epoch  40  lr=0.00010  train_acc=0.9881  val_top1=0.7311  val_mca=0.6378
[step20] early stopping at epoch 49, best val_top1=0.7426 (epoch 29)


███████████████████████████████ 100% | 11.66/11.66 MB [00:01<00:00,  6.57MB/s]: 


[step20] FINAL: best_epoch=29  test_top1=0.7072  gap=0.2808  test_mca=0.5785  test_macro_f1=0.6048


### comparison table

In [10]:
LI_ET_AL_BENCHMARK = {"config_name": "Li et al. benchmark", "step_size": None, "best_epoch": None,
                      "test_top1": 0.7441, "test_top5": 0.9376, "test_mca": 0.6144,
                      "test_macro_f1": None, "train_test_gap": None}

comparison = pd.DataFrame(list(all_results.values()) + [LI_ET_AL_BENCHMARK]).set_index("config_name")
comparison = comparison[["step_size", "best_epoch", "train_acc", "test_top1", "train_test_gap",
                          "test_top5", "test_mca", "test_macro_f1"]]
comparison.to_csv(os.path.join(OUT_ROOT, "schedule_comparison.csv"))
comparison


,step_size,best_epoch,train_acc,test_top1,train_test_gap,test_top5,test_mca,test_macro_f1
config_name,,,,,,,,
step10,10.0,27.0,0.927052,0.710997,0.216054,0.945013,0.615995,0.630229
step15,15.0,50.0,0.993921,0.709719,0.284202,0.942455,0.590176,0.611574
step20,20.0,29.0,0.988002,0.707161,0.280841,0.936061,0.578451,0.604830
Li et al. benchmark,NaN,NaN,NaN,0.744100,NaN,0.937600,0.614400,NaN


---

Check `best_epoch` for each first - if step15/step20 push the peak meaningfully later than step10's, that confirms the schedule really was cutting training short. Then check whether test_top1 actually improved alongside it, not just moved the peak later without raising it - a later peak isn't automatically a better one.